<a href="https://colab.research.google.com/github/souvikdey214-code/Krutanic_Weekly-Project-Task/blob/main/12_TalentLens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pdfminer.six sentence-transformers spacy fastapi "uvicorn[standard]" python-multipart reportlab pandas requests
!python -m spacy download en_core_web_lg -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from pathlib import Path

_RAW = """
python, java, javascript, typescript, c++, c#, golang, rust, kotlin, swift, scala, ruby, php, matlab, sql, bash,
shell scripting, perl, dart, julia, r programming, html, css, sass, solidity, haskell, lua, objective-c, groovy,
powershell, assembly, verilog, vhdl, fortran, cobol, c programming, pl/sql, t-sql, vba,
machine learning, deep learning, nlp, natural language processing, computer vision, reinforcement learning,
neural networks, cnn, rnn, lstm, gru, transformers, bert, gpt, llm, large language models, generative ai,
prompt engineering, rag, retrieval augmented generation, langchain, llamaindex, hugging face, fine-tuning,
transfer learning, scikit-learn, sklearn, tensorflow, keras, pytorch, jax, xgboost, lightgbm, catboost, opencv,
yolo, spacy, nltk, gensim, mlflow, kubeflow, mlops, feature engineering, hyperparameter tuning, time series,
forecasting, recommender systems, anomaly detection, clustering, regression, classification, k-means,
random forest, decision trees, svm, naive bayes, gradient boosting, pca, dimensionality reduction, gan, autoencoder,
object detection, image segmentation, image classification, sentiment analysis, named entity recognition,
speech recognition, text classification, topic modeling, word2vec, sentence transformers, sbert, embeddings,
vector database, faiss, pinecone, chroma, weaviate, onnx, tensorrt, model deployment, model serving,
explainable ai, shap, lime, grad-cam, federated learning, graph neural networks, bayesian inference,
pandas, numpy, scipy, matplotlib, seaborn, plotly, tableau, power bi, excel, google sheets, looker, data analysis,
data visualization, data engineering, data science, data mining, data cleaning, data modeling, data warehousing,
etl, elt, spark, pyspark, hadoop, kafka, airflow, dbt, snowflake, databricks, bigquery, redshift, hive, flink,
statistics, hypothesis testing, a/b testing, probability, linear algebra, calculus, big data, business intelligence,
fastapi, flask, django, node.js, express, react, angular, vue, next.js, nuxt, svelte, spring boot, spring,
rest api, restful apis, graphql, grpc, microservices, websockets, tailwind css, bootstrap, redux, jquery, webpack,
.net, asp.net, laravel, rails, ruby on rails, streamlit, gradio, celery, rabbitmq, oauth, jwt, api design,
mysql, postgresql, mongodb, redis, sqlite, oracle, cassandra, elasticsearch, dynamodb, neo4j, firebase,
sql server, mariadb, couchdb, influxdb, supabase, orm, sqlalchemy, prisma,
aws, azure, gcp, google cloud, docker, kubernetes, terraform, ansible, jenkins, github actions, gitlab ci, ci/cd,
git, github, gitlab, bitbucket, linux, unix, nginx, apache, prometheus, grafana, ec2, s3, lambda, sagemaker,
vertex ai, azure ml, cloudformation, helm, serverless, devops, sre, load balancing, monitoring, logging,
cybersecurity, network security, intrusion detection, ids, ips, penetration testing, cryptography, siem, owasp,
vulnerability assessment, malware analysis, firewall, vpn, wireshark, nmap, metasploit, blockchain, vanet,
wireless sensor networks, iot, embedded systems, arduino, raspberry pi, fpga, rtos, mqtt, ros, robotics,
android, ios, flutter, react native, swiftui, jetpack compose, xcode, android studio, unity, unreal engine, opengl,
agile, scrum, kanban, jira, confluence, postman, swagger, figma, ui/ux, product management, project management,
unit testing, pytest, junit, selenium, cypress, test automation, tdd, code review, system design,
design patterns, data structures, algorithms, object-oriented programming, functional programming,
distributed systems, concurrency, multithreading, parallel computing, cuda, hpc,
leadership, communication, problem solving, teamwork, mentoring, public speaking, technical writing,
research, publications, latex, jupyter, google colab, kaggle, vs code, intellij, pycharm
"""
SKILLS = list(dict.fromkeys(s.strip().lower() for s in _RAW.replace("\n", " ").split(",") if s.strip()))
Path("skills_taxonomy.py").write_text("SKILLS = " + repr(SKILLS) + "\n")
print(f"{len(SKILLS)} skills saved to skills_taxonomy.py")
print("The curriculum suggests 500+ - add more domain-specific skills to the list above as you go.")

346 skills saved to skills_taxonomy.py
The curriculum suggests 500+ - add more domain-specific skills to the list above as you go.


In [ ]:
%%writefile screener.py
"""TalentLens screening engine: PDF parsing, skill extraction, SBERT matching, scoring."""
import re
import numpy as np
import spacy
from spacy.matcher import PhraseMatcher
from pdfminer.high_level import extract_text as _pdf_extract_text
from pdfminer.layout import LAParams
from sentence_transformers import SentenceTransformer, util

from skills_taxonomy import SKILLS

nlp = spacy.load("en_core_web_lg")
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
_matcher.add("SKILLS", [nlp.make_doc(s) for s in SKILLS])

WEIGHTS = {"skills": 0.4, "semantic": 0.4, "experience": 0.2}
EXPERIENCE_BENCHMARK_YEARS = 5


# ---------- Step 1: resume parsing ----------
def clean_text(text: str) -> str:
    text = text.replace("\x0c", "\n").replace("•", " ").replace("●", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text(pdf_path: str) -> str:
    """Extract and clean text from a PDF. LAParams groups text blocks, which copes with multi-column layouts."""
    laparams = LAParams(line_margin=0.5, boxes_flow=0.5)
    return clean_text(_pdf_extract_text(pdf_path, laparams=laparams))


# ---------- Step 2: skills / experience / name ----------
def extract_skills(text: str) -> set:
    doc = nlp.make_doc(text)                       # tokenisation is all PhraseMatcher needs (fast)
    return {" ".join(doc[s:e].text.lower().split()) for _, s, e in _matcher(doc)}


def extract_years(text: str) -> int:
    nums = [int(n) for n in re.findall(r"(\d+)\+?\s*year[s]?", text.lower())]
    nums = [n for n in nums if n <= 40]            # ignore silly values like "100 years"
    return max(nums, default=0)


def extract_candidate_name(text: str, fallback: str = "Unknown") -> str:
    for ent in nlp(text[:600]).ents:               # NER on the top of the resume
        if ent.label_ == "PERSON" and len(ent.text.split()) >= 2:
            return ent.text.strip()
    return fallback


# ---------- Step 3: semantic similarity ----------
def embed(text: str, chunk_words: int = 200):
    words = text.split()
    chunks = [" ".join(words[i:i + chunk_words]) for i in range(0, max(len(words), 1), chunk_words)] or [""]
    vecs = sbert.encode(chunks, convert_to_tensor=True, normalize_embeddings=True)
    return vecs.mean(dim=0)                        # one 384-d vector


def semantic_similarity(a: str, b: str) -> float:
    return max(0.0, float(util.cos_sim(embed(a), embed(b))))


# ---------- Step 4: scoring ----------
def screen(resume_text: str, jd_text: str, fallback_name: str = "Unknown") -> dict:
    r_sk, j_sk = extract_skills(resume_text), extract_skills(jd_text)
    skill_overlap = len(r_sk & j_sk) / max(len(j_sk), 1)
    sbert_sim = semantic_similarity(resume_text, jd_text)
    years = extract_years(resume_text)
    exp_match = min(years / EXPERIENCE_BENCHMARK_YEARS, 1.0)

    final = (WEIGHTS["skills"] * skill_overlap + WEIGHTS["semantic"] * sbert_sim + WEIGHTS["experience"] * exp_match)
    score = round(final * 100, 2)
    recommendation = "Shortlist" if score >= 70 else "Review" if score >= 40 else "Reject"
    return {
        "candidate_name": extract_candidate_name(resume_text, fallback_name),
        "match_score": score,
        "matched_skills": sorted(r_sk & j_sk),
        "missing_skills": sorted(j_sk - r_sk),
        "recommendation": recommendation,
        "details": {"skill_overlap": round(skill_overlap, 3), "semantic_similarity": round(sbert_sim, 3),
                    "years_experience": years, "experience_match": round(exp_match, 3)},
    }

Writing screener.py


In [ ]:
import importlib, screener
importlib.reload(screener)

resume = "Aisha Khan. ML Engineer with 4 years of experience in Python, PyTorch, NLP and Docker. Built FastAPI services on AWS."
jd = "We need a machine learning engineer: Python, PyTorch, NLP, Docker, Kubernetes, SQL. 3+ years experience."

print("Resume skills :", sorted(screener.extract_skills(resume)))
print("JD skills     :", sorted(screener.extract_skills(jd)))
print("Years         :", screener.extract_years(resume))
print("Similarity    :", round(screener.semantic_similarity(resume, jd), 3))
screener.screen(resume, jd)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Resume skills : ['aws', 'docker', 'fastapi', 'nlp', 'python', 'pytorch']
JD skills     : ['docker', 'kubernetes', 'machine learning', 'nlp', 'python', 'pytorch', 'sql']
Years         : 4
Similarity    : 0.591


{'candidate_name': 'Aisha Khan',
 'match_score': 62.5,
 'matched_skills': ['docker', 'nlp', 'python', 'pytorch'],
 'missing_skills': ['kubernetes', 'machine learning', 'sql'],
 'recommendation': 'Review',
 'details': {'skill_overlap': 0.571,
  'semantic_similarity': 0.591,
  'years_experience': 4,
  'experience_match': 0.8}}

In [ ]:
%%writefile app.py
import os
import tempfile
from pathlib import Path

from fastapi import FastAPI, File, Form, HTTPException, UploadFile

import screener

app = FastAPI(title="TalentLens - Resume Screener API", version="1.0.0",
              description="Upload a resume PDF and a job description; get a match score, matched/missing skills and a recommendation.")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/screen")
async def screen_resume(resume: UploadFile = File(..., description="Resume as a PDF"),
                        job_description: str = Form(..., description="Job description text")):
    if not (resume.filename or "").lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Please upload a PDF file.")
    if not job_description.strip():
        raise HTTPException(status_code=400, detail="job_description must not be empty.")

    data = await resume.read()
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
        tmp.write(data)
        tmp_path = tmp.name
    try:
        text = screener.extract_text(tmp_path)
    except Exception as exc:
        raise HTTPException(status_code=422, detail=f"Could not read the PDF: {exc}")
    finally:
        os.remove(tmp_path)

    if len(text) < 50:
        raise HTTPException(status_code=422, detail="No readable text found (scanned image PDFs need OCR).")
    return screener.screen(text, job_description, fallback_name=Path(resume.filename).stem)

Writing app.py


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
import os
os.makedirs("samples", exist_ok=True)

RESUMES = {
 "Aisha_Khan_ML": ["Aisha Khan", "Machine Learning Engineer", "",
   "Summary: 4 years of experience building NLP and computer vision models in production.",
   "Skills: Python, PyTorch, TensorFlow, NLP, transformers, hugging face, Docker, FastAPI, AWS, SQL, git",
   "Experience: Built an intent-classification service with BERT; deployed models with Docker and FastAPI on AWS.",
   "Education: B.Tech Computer Science"],
 "Rahul_Verma_Web": ["Rahul Verma", "Full-Stack Developer", "",
   "Summary: 3 years of experience building web apps.",
   "Skills: JavaScript, TypeScript, React, Node.js, express, MongoDB, PostgreSQL, REST API, git, Docker",
   "Experience: Built dashboards in React and REST APIs in Node.js for an e-commerce company.",
   "Education: B.Sc Computer Science"],
 "Priya_Das_Data": ["Priya Das", "Data Analyst", "",
   "Summary: 2 years of experience in analytics and reporting.",
   "Skills: SQL, Excel, Tableau, Power BI, pandas, numpy, statistics, python, data visualization",
   "Experience: Built weekly KPI dashboards and ran A/B tests for a retail business.",
   "Education: B.Com, Data Analytics certificate"],
}
for fname, lines in RESUMES.items():
    c = canvas.Canvas(f"samples/{fname}.pdf", pagesize=A4); y = 800
    for ln in lines:
        c.drawString(50, y, ln); y -= 22
    c.save()

JOB_DESCRIPTIONS = {
 "ML Engineer": "Machine learning engineer to build NLP models. Required: Python, PyTorch, NLP, transformers, Docker, FastAPI, AWS, SQL. 3+ years experience.",
 "Full-Stack Developer": "Full-stack developer. Required: JavaScript, TypeScript, React, Node.js, PostgreSQL, REST API, Docker, git. 3 years experience.",
 "Data Analyst": "Data analyst for business reporting. Required: SQL, Excel, Tableau, Power BI, statistics, python, data visualization. 2 years experience.",
}
print("Created:", sorted(os.listdir("samples")))

Created: ['Aisha_Khan_ML.pdf', 'Priya_Das_Data.pdf', 'Rahul_Verma_Web.pdf']


In [ ]:
import subprocess, time, requests

server = subprocess.Popen(["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
print("Starting API (loading the spaCy + SBERT models takes a minute)...")
for _ in range(180):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).json().get("status") == "ok":
            print("✅ API is up:", requests.get("http://localhost:8000/health").json()); break
    except Exception:
        time.sleep(2)
else:
    print("API did not start - check server.log:\n", open("server.log").read()[-1500:])

Starting API (loading the spaCy + SBERT models takes a minute)...
✅ API is up: {'status': 'ok'}


In [ ]:
# Call the API exactly as a client (or Postman) would
with open("samples/Aisha_Khan_ML.pdf", "rb") as f:
    r = requests.post("http://localhost:8000/screen",
                      files={"resume": ("Aisha_Khan_ML.pdf", f, "application/pdf")},
                      data={"job_description": JOB_DESCRIPTIONS["ML Engineer"]})
print("HTTP", r.status_code)
import json; print(json.dumps(r.json(), indent=2))

HTTP 200
{
  "candidate_name": "Aisha_Khan_ML",
  "match_score": 84.9,
  "matched_skills": [
    "aws",
    "docker",
    "fastapi",
    "machine learning",
    "nlp",
    "python",
    "pytorch",
    "sql",
    "transformers"
  ],
  "missing_skills": [],
  "recommendation": "Shortlist",
  "details": {
    "skill_overlap": 1.0,
    "semantic_similarity": 0.723,
    "years_experience": 4,
    "experience_match": 0.8
  }
}


In [ ]:
import pandas as pd, glob

rows = []
for pdf in sorted(glob.glob("samples/*.pdf")):
    for role, jd in JOB_DESCRIPTIONS.items():
        with open(pdf, "rb") as f:
            res = requests.post("http://localhost:8000/screen", files={"resume": (os.path.basename(pdf), f, "application/pdf")},
                                data={"job_description": jd}).json()
        rows.append({"resume": os.path.basename(pdf).replace(".pdf", ""), "job": role,
                     "score": res["match_score"], "recommendation": res["recommendation"]})
df = pd.DataFrame(rows)
display(df.pivot(index="resume", columns="job", values="score"))
df.sort_values("score", ascending=False).head(9)

job,Data Analyst,Full-Stack Developer,ML Engineer
resume,,,
Aisha_Khan_ML,42.32,40.82,84.90
Priya_Das_Data,78.20,19.54,28.57
Rahul_Verma_Web,28.76,83.87,31.40


,resume,job,score,recommendation
0,Aisha_Khan_ML,ML Engineer,84.90,Shortlist
7,Rahul_Verma_Web,Full-Stack Developer,83.87,Shortlist
5,Priya_Das_Data,Data Analyst,78.20,Shortlist
2,Aisha_Khan_ML,Data Analyst,42.32,Review
1,Aisha_Khan_ML,Full-Stack Developer,40.82,Review
6,Rahul_Verma_Web,ML Engineer,31.40,Reject
8,Rahul_Verma_Web,Data Analyst,28.76,Reject
3,Priya_Das_Data,ML Engineer,28.57,Reject
4,Priya_Das_Data,Full-Stack Developer,19.54,Reject


In [ ]:
from google.colab import files
import json, screener

uploaded = files.upload()                                   # choose a resume PDF
resume_path = next(iter(uploaded))
jd_text = input("Paste the job description: ")

resume_text = screener.extract_text(resume_path)
result = screener.screen(resume_text, jd_text, fallback_name=os.path.splitext(resume_path)[0])

print(f"\nCandidate      : {result['candidate_name']}")
print(f"Match score    : {result['match_score']} / 100   ->  {result['recommendation'].upper()}")
print(f"Matched skills : {', '.join(result['matched_skills']) or '-'}")
print(f"Missing skills : {', '.join(result['missing_skills']) or '-'}")
print("\nFull JSON:\n", json.dumps(result, indent=2))

Saving SD_Resume.pdf to SD_Resume.pdf
Paste the job description: Ai Engineer worker.

Candidate      : deep learni
Match score    : 14.89 / 100   ->  REJECT
Matched skills : -
Missing skills : -

Full JSON:
 {
  "candidate_name": "deep learni",
  "match_score": 14.89,
  "matched_skills": [],
  "missing_skills": [],
  "recommendation": "Reject",
  "details": {
    "skill_overlap": 0.0,
    "semantic_similarity": 0.372,
    "years_experience": 0,
    "experience_match": 0.0
  }
}


In [ ]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
python-multipart
pdfminer.six
spacy
sentence-transformers

Writing requirements.txt


In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt \
    && python -m spacy download en_core_web_lg \
    && python -c "from sentence_transformers import SentenceTransformer; SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')"

COPY app.py screener.py skills_taxonomy.py ./

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [ ]:
import zipfile
from google.colab import files

server.terminate()                                          # stop the background API
with zipfile.ZipFile("talentlens_project.zip", "w") as z:
    for fn in ["app.py", "screener.py", "skills_taxonomy.py", "requirements.txt", "Dockerfile"]:
        z.write(fn)
    for pdf in glob.glob("samples/*.pdf"):
        z.write(pdf)
files.download("talentlens_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>